# Comparative Analysis of Custom Local-Search AutoML vs. Industry Standards (FLAML & TPOT)

## Project Overview
This notebook presents a comparative study of a custom-built **Local Search AutoML engine** against two widely adopted, industry-grade frameworks: **FLAML** (Fast and Lightweight AutoML) and **TPOT** (Tree-based Pipeline Optimization Tool). The evaluation benchmark tests all three systems across two distinct regression tasks under a strictly constrained uniform time budget of **60 minutes (3,600 seconds) per system per run**:
1. **Phone Prices Prediction**: Estimating phone market value utilizing continuous hardware and sparse temporal specifications.
2. **Coffee Shop Revenue Forecasting**: Time-series-adjacent operational analytics mapping storefront features to daily sales revenue.

## Experimental Methodology
To maintain rigorous experimental control and avoid structural leakage, all systems share an identical data pipeline layout:
* **Preprocessing Pipeline**: Continuous traits are imputed via sample medians; categorical fields undergo robust handle-unknown-safe One-Hot Encoding (OHE).
* **Cross-Validation Framework**: Hyperparameter optimization sets its optimization objective against a unified internal Cross-Validation Root Mean Squared Error (CV RMSE).
* **Generalization Assessment**: Final performance measurements are collected from isolated out-of-sample Test sets via Holdout Test RMSE.

---

## 1. System Baseline & Environment Setup

### Environment Validation

In [3]:
# Verifying runtime consistency across execution environments
import sklearn, sys
print("sklearn:", sklearn.__version__)
print("python:", sys.version)

sklearn: 1.8.0
python: 3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]


### Essential Packages Import

In [5]:
from __future__ import annotations

import time
import math
import random
from dataclasses import dataclass
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split

## 2. Feature Engineering & Dataset Diagnostics

Before routing data into the search arrays, features must be parsed cleanly:
* **Temporal Discretization**: High-cardinality date fields are decomposed into independent cyclical seasonal components (`announcement_year`, `announcement_month`).
* **Text Resolution Decomposition**: Combined screen resolution strings (e.g., `1080x2400`) are tokenized into spatial coordinates (`res_width`, `res_height`) and scaled into a global area metric (`res_total_pixels`).
* **Boolean Normalization**: Varied string/boolean representations of multimedia attributes are cast into standard binary integers.

### Code Block 3: Multi-Dataset Extraction, Cleaning, and Matrix Splits

In [7]:
df_phone = pd.read_csv("cleaned_all_phones.csv")
df_coffee = pd.read_csv("coffee_shop_revenue.csv")

# -------------------------
# PHONE DATA CLEANING
# -------------------------

# Drop phone_name
df_phone = df_phone.drop(columns=["phone_name"], errors="ignore")

# Robust video columns conversion
video_cols = [c for c in df_phone.columns if c.startswith("video_")]
for c in video_cols:
    if df_phone[c].dtype == bool:
        df_phone[c] = df_phone[c].astype(int)
    else:
        df_phone[c] = (
            df_phone[c].astype(str).str.lower()
            .map({"true": 1, "false": 0})
        )

# Resolution to numeric
res = df_phone["resolution"].astype(str)
split = res.str.split("x", n=1, expand=True)
df_phone["res_width"]  = pd.to_numeric(split[0], errors="coerce")
df_phone["res_height"] = pd.to_numeric(split[1], errors="coerce")
df_phone["res_total_pixels"] = df_phone["res_width"] * df_phone["res_height"]
df_phone = df_phone.drop(columns=["resolution"], errors="ignore")

# Date to year/month
dt = pd.to_datetime(df_phone["announcement_date"], errors="coerce")
df_phone["announcement_year"] = dt.dt.year
df_phone["announcement_month"] = dt.dt.month
df_phone = df_phone.drop(columns=["announcement_date"], errors="ignore")

# -------------------------
# SPLIT FEATURES / TARGET
# -------------------------

X_phone = df_phone.drop(columns=["price(USD)"])
y_phone = df_phone["price(USD)"]

X_coffee = df_coffee.drop(columns=["Daily_Revenue"])
y_coffee = df_coffee["Daily_Revenue"]

## 3. Custom AutoML Engine: Random-Restart Hill Climbing

The custom engine implements a discrete meta-heuristic optimization algorithm: **Random-Restart Hill Climbing**. 

### Structural Mechanics:
1. **Discrete Coordinate Search Space**: Restricts hyperparameters to explicitly defined stepping rings, enabling logical neighborhood hops (`±1` steps).
2. **Algorithmic Adaptability**: Optimizes across four foundational model paradigms: Random Forests (`rf`), Support Vector Regressors (`svr`), K-Nearest Neighbors (`knn`), and Gradient Boosting Machines (`gbr`).
3. **Pipeline Augmentation Factory**: Dynamically scales column outputs if the sampled candidate model is distance-dependent (`knn` or `svr`), protecting standard tree algorithms from structural scaling distortions.
4. **Local Minimum Mitigation**: Executes multiple randomized search vectors within the time budget to avoid localized convergence traps.

### Object-Oriented Framework for Local Search AutoML Engine

In [12]:
@dataclass(frozen=True)
class Config:
    algo: str
    params: Dict[str, Any]


class LocalSearchAutoML:
    """
    Hill-climbing with random restarts over (algorithm, hyperparameters).
    - Discrete parameter domains to support +/-1 neighbor moves.
    - CV RMSE objective.
    """

    def __init__(
        self,
        numeric_features: Optional[List[str]] = None,
        categorical_features: Optional[List[str]] = None,
        cv_splits: int = 5,
        seed: int = 42,
    ):
        self.numeric_features = numeric_features
        self.categorical_features = categorical_features
        self.cv_splits = cv_splits
        self.rng = random.Random(seed)
        self.seed = seed

        # Define discrete search space
        self.space = {
            "rf": {
                "n_estimators": [50, 100, 200, 400],
                "max_features": [0.3, 0.5, 0.7, 1.0],
                "max_depth": [None, 5, 10, 20],
                "min_samples_split": [2, 5, 10],
            },
            "svr": {
                "C": [0.1, 1.0, 10.0, 100.0],
                "epsilon": [0.01, 0.1, 0.2],
                "gamma": ["scale", "auto"],
                "kernel": ["rbf"],  # keep fixed to simplify
            },
            "knn": {
                "n_neighbors": [3, 5, 7, 9, 15, 25],
                "weights": ["uniform", "distance"],
                "p": [1, 2],
            },
            "gbr": {
                "n_estimators": [100, 200, 400],
                "learning_rate": [0.01, 0.05, 0.1],
                "max_depth": [2, 3, 4],
                "subsample": [0.7, 1.0],
            },
        }

    # ----------------------------
    # Model / pipeline factory
    # ----------------------------
    def _build_preprocessor(self, X):
    # X is already preprocessed numeric matrix (numpy/sparse) by your shared preprocessor
        return FunctionTransformer(lambda Z: Z, validate=False)
    
    def _estimator_from_config(self, cfg: Config) -> BaseEstimator:
        if cfg.algo == "rf":
            return RandomForestRegressor(random_state=self.seed, n_jobs=-1, **cfg.params)
        if cfg.algo == "svr":
            return SVR(**cfg.params)
        if cfg.algo == "knn":
            return KNeighborsRegressor(**cfg.params)
        if cfg.algo == "gbr":
            return GradientBoostingRegressor(random_state=self.seed, **cfg.params)
        raise ValueError(f"Unknown algo: {cfg.algo}")

    def _pipeline_from_config(self, cfg: Config, X) -> Pipeline:
        pre = self._build_preprocessor(X)

        # Add scaling for algorithms that are distance / margin based
        needs_scaling = cfg.algo in {"svr", "knn"}
        if needs_scaling:
            # If preprocessor is a ColumnTransformer, scale numeric output only is tricky after OHE.
            # For simplicity: scale entire transformed matrix (works with sparse too via with_mean=False).
            scaler = StandardScaler(with_mean=False)
            pipe = Pipeline([("preprocess", pre), ("scale", scaler), ("model", self._estimator_from_config(cfg))])
        else:
            pipe = Pipeline([("preprocess", pre), ("model", self._estimator_from_config(cfg))])
        return pipe

    # ----------------------------
    # Sampling / neighbors
    # ----------------------------
    def sample_random_config(self) -> Config:
        algo = self.rng.choice(list(self.space.keys()))
        params = {}
        for hp, values in self.space[algo].items():
            params[hp] = self.rng.choice(values)
        return Config(algo=algo, params=params)

    def neighbors(self, cfg: Config) -> List[Config]:
        neigh = []

        # 1) Change algorithm (keep default/random params for the new algo)
        for other_algo in self.space.keys():
            if other_algo == cfg.algo:
                continue
            # create a "mapped" neighbor by randomizing params of other algo
            params = {hp: self.rng.choice(vals) for hp, vals in self.space[other_algo].items()}
            neigh.append(Config(other_algo, params))

        # 2) +/-1 step for one hyperparameter at a time (discrete)
        for hp, values in self.space[cfg.algo].items():
            current_val = cfg.params[hp]
            idx = values.index(current_val)
            for delta in (-1, +1):
                j = idx + delta
                if 0 <= j < len(values):
                    new_params = dict(cfg.params)
                    new_params[hp] = values[j]
                    neigh.append(Config(cfg.algo, new_params))

        # Deduplicate
        uniq = []
        seen = set()
        for n in neigh:
            key = (n.algo, tuple(sorted(n.params.items())))
            if key not in seen:
                seen.add(key)
                uniq.append(n)
        return uniq

    # ----------------------------
    # Evaluation
    # ----------------------------
    def score_config(self, cfg: Config, X, y) -> float:
        y = np.asarray(y).ravel()
        pipe = self._pipeline_from_config(cfg, X)
        cv = KFold(n_splits=self.cv_splits, shuffle=True, random_state=self.seed)
        y_pred = cross_val_predict(pipe, X, y, cv=cv, n_jobs=1)
        return rmse(y, y_pred)

    # ----------------------------
    # Main search
    # ----------------------------
    def fit(
        self,
        X,
        y,
        time_budget_seconds: int = 3600,
        max_no_improve_steps: int = 25,
        restarts: int = 25,
        verbose: bool = True,
    ) -> Dict[str, Any]:
        start = time.time()

        best_cfg = None
        best_score = float("inf")
        history: List[Tuple[float, Config, float]] = []  # (t, cfg, score)

        def time_left() -> float:
            return time_budget_seconds - (time.time() - start)

        for r in range(restarts):
            if time_left() <= 0:
                break

            current = self.sample_random_config()
            try:
                current_score = self.score_config(current, X, y)
            except Exception as e:
                if verbose:
                    print("[eval failed]", current, repr(e))
                continue

            if current_score < best_score:
                best_score, best_cfg = current_score, current
            history.append((time.time() - start, current, current_score))

            if verbose:
                print(f"[restart {r+1}/{restarts}] init {current.algo} score={current_score:.4f} best={best_score:.4f}")

            no_improve = 0
            while time_left() > 0 and no_improve < max_no_improve_steps:
                neigh = self.neighbors(current)
                self.rng.shuffle(neigh)

                improved = False
                best_local = (current_score, current)

                for cand in neigh:
                    if time_left() <= 0:
                        break
                    try:
                        s = self.score_config(cand, X, y)
                    except Exception as e:
                        if verbose:
                            print("[neighbor eval failed]", cand, repr(e))
                        continue
                    history.append((time.time() - start, cand, s))

                    if s < best_local[0]:
                        best_local = (s, cand)
                    if s < best_score:
                        best_score, best_cfg = s, cand

                if best_local[0] < current_score:
                    current_score, current = best_local
                    improved = True
                    no_improve = 0
                    if verbose:
                        print(f"  -> move to {current.algo} score={current_score:.4f} best={best_score:.4f}")
                else:
                    no_improve += 1
                    if not improved and verbose:
                        print(f"  (no improve step {no_improve}/{max_no_improve_steps})")
        # Fallback if no configuration was successfully evaluated
        if best_cfg is None:
            best_cfg = Config(
                "rf",
                {
                    "n_estimators": 200,
                    "max_features": 1.0,
                    "max_depth": None,
                    "min_samples_split": 2,
                },
            )
            try:
                best_score = self.score_config(best_cfg, X, y)
            except Exception:
                pass
                
        return {
            "best_config": best_cfg,
            "best_rmse": best_score,
            "history": history,
            "time_used_seconds": time.time() - start,
        }

## 4. Standardized Cross-Validation Execution Engine

To ensure an unbiased evaluation, this block abstracts execution logic away from framework-specific APIs. It forces a shared data split configuration and isolates out-of-sample data points until final evaluation testing.

### Automated Preprocessing and Execution Driver Functions

In [14]:
def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

# 0) Shared prep

def prepare_train_test_with_shared_preprocessing(
    X: pd.DataFrame,
    y,
    seed: int = 42,
    test_size: float = 0.2,
    scale_after_ohe: bool = False, 
):
    """
    One split + one preprocessing for ALL systems.
    Returns numeric matrices suitable for our method, FLAML, and TPOT.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=seed
    )

    num_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
    cat_cols = [c for c in X_train.columns if c not in num_cols]

    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ])

    pre = ColumnTransformer(
        transformers=[
            ("num", num_pipe, num_cols),
            ("cat", cat_pipe, cat_cols),
        ],
        remainder="drop",
    )

    X_train_pre = pre.fit_transform(X_train)
    X_test_pre  = pre.transform(X_test)

    if scale_after_ohe:
        scaler = StandardScaler(with_mean=False)
        X_train_pre = scaler.fit_transform(X_train_pre)
        X_test_pre  = scaler.transform(X_test_pre)

    y_train_np = np.asarray(y_train).ravel()
    y_test_np  = np.asarray(y_test).ravel()

    return X_train_pre, X_test_pre, y_train_np, y_test_np, pre

# 1) Our local-search AutoML

def run_my_automl(
    X_train_pre, y_train_np,
    X_test_pre, y_test_np,
    dataset_name: str,
    time_budget_seconds: int = 3600,
    seed: int = 42,
    cv_splits: int = 5,
    verbose: bool = True,
):
    my_automl = LocalSearchAutoML(cv_splits=cv_splits, seed=seed)

    t0 = time.time()
    my_res = my_automl.fit(
        X_train_pre, y_train_np,
        time_budget_seconds=time_budget_seconds,
        verbose=verbose
    )
    my_time = time.time() - t0

    best_cfg = my_res["best_config"]
    best_pipe = my_automl._pipeline_from_config(best_cfg, X_train_pre)
    best_pipe.fit(X_train_pre, y_train_np)
    y_pred = best_pipe.predict(X_test_pre)

    out_row = {
        "dataset": dataset_name,
        "system": "MyAutoML_LocalSearch",
        "test_rmse": rmse(y_test_np, y_pred),
        "time_seconds": my_time,
        "details": str(best_cfg),
    }

    hist = pd.DataFrame(
        [{"t": t, "algo": cfg.algo, "cv_rmse": score} for (t, cfg, score) in my_res["history"]]
    )
    if len(hist) > 0:
        hist["best_so_far"] = hist["cv_rmse"].cummin()

    return out_row, hist


# 2) FLAML

def run_flaml(
    X_train_pre, y_train_np,
    X_test_pre, y_test_np,
    dataset_name: str,
    time_budget_seconds: int = 3600,
    seed: int = 42,
    verbose: int = 1,
):
    from flaml import AutoML

    t0 = time.time()
    automl = AutoML()
    automl.fit(
        X_train_pre, y_train_np,
        task="regression",
        metric="rmse",
        time_budget=time_budget_seconds,
        verbose=verbose,
        seed=seed,
    )
    wall = time.time() - t0

    y_pred = automl.predict(X_test_pre)

    out_row = {
        "dataset": dataset_name,
        "system": "FLAML",
        "test_rmse": rmse(y_test_np, y_pred),
        "time_seconds": wall,
        "details": f"best_estimator={automl.best_estimator}",
    }
    return out_row


# 3) TPOT (v1.1.0) 

def run_tpot(
    X_train_pre, y_train_np,
    X_test_pre, y_test_np,
    dataset_name: str,
    time_budget_seconds: int = 3600,
    seed: int = 42,
    verbose: int = 2,
    search_space: str = "linear-light",
):
    """
    Uses TPOT 1.1.0 style args. We set n_jobs=1 to avoid Dask worker issues on macOS.
    """
    from tpot import TPOTRegressor

    t0 = time.time()
    tpot = TPOTRegressor(
        search_space=search_space,
        scorers=["neg_root_mean_squared_error"],
        scorers_weights=[1.0],
        max_time_mins=max(1, time_budget_seconds // 60),
        random_state=seed,
        verbose=verbose,
        n_jobs=1, 
    )
    tpot.fit(X_train_pre, y_train_np)
    wall = time.time() - t0

    y_pred = tpot.predict(X_test_pre)

    out_row = {
        "dataset": dataset_name,
        "system": "TPOT",
        "test_rmse": rmse(y_test_np, y_pred),
        "time_seconds": wall,
        "details": f"search_space={search_space}",
    }
    return out_row

## Case Study 1 — Phone Prices Dataset

### Phone Prices Train-Test Coordinate Matrix Generation

In [26]:
X_train_pre, X_test_pre, y_train_np, y_test_np, pre = prepare_train_test_with_shared_preprocessing(
    X_phone, y_phone, seed=42, scale_after_ohe=False
)

### Runner 1.1: Custom Local Search Execution (Phone Prices)

In [28]:
my_row, my_hist = run_my_automl(
    X_train_pre, y_train_np, X_test_pre, y_test_np,
    dataset_name="PhonePrices",
    time_budget_seconds=3600,
    seed=42
)
my_row

[restart 1/25] init rf score=229.0714 best=229.0714
  -> move to rf score=227.8596 best=227.8596
  -> move to rf score=227.6436 best=227.6436
  -> move to rf score=226.6856 best=226.6856
  -> move to rf score=225.3508 best=225.3508
  -> move to rf score=223.7171 best=223.7171
  -> move to rf score=222.8010 best=222.8010
  (no improve step 1/25)
  (no improve step 2/25)
  (no improve step 3/25)
  (no improve step 4/25)
  (no improve step 5/25)
  (no improve step 6/25)
  (no improve step 7/25)
  (no improve step 8/25)
  (no improve step 9/25)
  (no improve step 10/25)
  (no improve step 11/25)
  (no improve step 12/25)
  (no improve step 13/25)
  (no improve step 14/25)
  (no improve step 15/25)
  (no improve step 16/25)
  (no improve step 17/25)
  (no improve step 18/25)
  (no improve step 19/25)
  (no improve step 20/25)
  (no improve step 21/25)
  (no improve step 22/25)
  (no improve step 23/25)
  (no improve step 24/25)
  (no improve step 25/25)
[restart 2/25] init gbr score=230.140

{'dataset': 'PhonePrices',
 'system': 'MyAutoML_LocalSearch',
 'test_rmse': 225.1479957189883,
 'time_seconds': 3600.6470868587494,
 'details': "Config(algo='rf', params={'n_estimators': 100, 'max_features': 0.3, 'max_depth': 10, 'min_samples_split': 10})"}

### Runner 1.2: FLAML Framework Optimization Process (Phone Prices)

In [23]:
!pip install flaml

In [30]:
flaml_row = run_flaml(
    X_train_pre, y_train_np, X_test_pre, y_test_np,
    dataset_name="PhonePrices",
    time_budget_seconds=3600,
    seed=42
)
flaml_row

/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-

{'dataset': 'PhonePrices',
 'system': 'FLAML',
 'test_rmse': 225.32525964907958,
 'time_seconds': 3600.711603164673,
 'details': 'best_estimator=xgb_limitdepth'}

### Runner 1.3: Genetic Pipeline Optimization via TPOT (Phone Prices)

In [32]:
tpot_row = run_tpot(
    X_train_pre, y_train_np, X_test_pre, y_test_np,
    dataset_name="PhonePrices",
    time_budget_seconds=3600,
    seed=42
)
tpot_row

Generation: : 14it [1:00:09, 257.79s/it]


{'dataset': 'PhonePrices',
 'system': 'TPOT',
 'test_rmse': 228.09823006816652,
 'time_seconds': 3611.743113040924,
 'details': 'search_space=linear-light'}

### Compile Framework Performance Metrics (Phone Prices)

In [34]:
results_df = pd.DataFrame([my_row, flaml_row, tpot_row]).sort_values("test_rmse")
results_df

,dataset,system,test_rmse,time_seconds,details
0,PhonePrices,MyAutoML_LocalSearch,225.147996,3600.647087,"Config(algo='rf', params={'n_estimators': 100,..."
1,PhonePrices,FLAML,225.325260,3600.711603,best_estimator=xgb_limitdepth
2,PhonePrices,TPOT,228.098230,3611.743113,search_space=linear-light


### Empirical Commentary: Phone Prices Optimization Results

| AutoML Framework | Selected Best Pipeline Model Type | Wall-Clock Execution Time (s) | Out-of-Sample Test RMSE |
| :--- | :--- | :--- | :--- |
| **MyAutoML_LocalSearch** | **Random Forest Regressor** (`n_estimators=100`, `max_features=0.3`, `max_depth=10`, `min_samples_split=10`) | **3600.65s** | **225.148** |
| **FLAML** | **LightGBM / XGBoost Variant** (`xgb_limitdepth`) | 3600.71s | 225.325 |
| **TPOT** | **Linear Pipeline Graph** (`linear-light` space) | 3611.74s | 228.098 |

#### Performance Analysis:
* Surprisingly, our custom local search optimizer outperformed both enterprise frameworks on this dataset, achieving the lowest generalization error (**Test RMSE: 225.148**). 
* The local search mechanism targeted a highly regularized Random Forest model with limited feature visibility (`max_features=0.3`, `max_depth=10`, `min_samples_split=10`). This strict regularized hyperparameter layout successfully mitigated overfitting on continuous hardware traits.
* FLAML closely matched this performance (**Test RMSE: 225.325**) by using an engineered XGBoost setup. TPOT lagged slightly behind (**Test RMSE: 228.098**), demonstrating that its genetic programming paradigm had a lower convergence velocity within the strict 60-minute cutoff when operating under the restricted `linear-light` space.

## Case Study 2 — Coffee Shop Dataset

### Coffee Shop Train-Test Coordinate Matrix Generation

In [18]:
X_train_pre, X_test_pre, y_train_np, y_test_np, pre = prepare_train_test_with_shared_preprocessing(
    X_coffee, y_coffee, seed=42, scale_after_ohe=False
)

### Runner 2.1: Custom Local Search Execution (Coffee Shop)

In [22]:
my_row, my_hist = run_my_automl(
    X_train_pre, y_train_np, X_test_pre, y_test_np,
    dataset_name="CoffeeShop",
    time_budget_seconds=3600,
    seed=42
)
my_row

[restart 1/25] init rf score=269.0528 best=269.0528
  -> move to rf score=233.7586 best=233.7586
  -> move to rf score=231.6550 best=231.6550
  -> move to rf score=231.0966 best=231.0966
  -> move to gbr score=226.2073 best=226.2073
  -> move to gbr score=220.7635 best=220.7635
  (no improve step 1/25)
  (no improve step 2/25)
  (no improve step 3/25)
  (no improve step 4/25)
  (no improve step 5/25)
  (no improve step 6/25)
  (no improve step 7/25)
  (no improve step 8/25)
  (no improve step 9/25)
  (no improve step 10/25)
  (no improve step 11/25)
  (no improve step 12/25)
  (no improve step 13/25)
  (no improve step 14/25)
  (no improve step 15/25)
  (no improve step 16/25)
  (no improve step 17/25)
  (no improve step 18/25)
  (no improve step 19/25)
  (no improve step 20/25)
  (no improve step 21/25)
  (no improve step 22/25)
  (no improve step 23/25)
  (no improve step 24/25)
  (no improve step 25/25)
[restart 2/25] init gbr score=226.7445 best=220.7635
  -> move to gbr score=223.

{'dataset': 'CoffeeShop',
 'system': 'MyAutoML_LocalSearch',
 'test_rmse': 214.31138335947693,
 'time_seconds': 3600.569636106491,
 'details': "Config(algo='gbr', params={'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 2, 'subsample': 0.7})"}

### Runner 2.2: FLAML Framework Optimization Process (Coffee Shop)

In [26]:
flaml_row = run_flaml(
    X_train_pre, y_train_np, X_test_pre, y_test_np,
    dataset_name="CoffeeShop",
    time_budget_seconds=3600,
    seed=42
)
flaml_row

/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.12/site-

{'dataset': 'CoffeeShop',
 'system': 'FLAML',
 'test_rmse': 205.64855794101885,
 'time_seconds': 3601.4041516780853,
 'details': 'best_estimator=catboost'}

### Runner 2.3: Genetic Pipeline Optimization via TPOT (Coffee Shop)

In [28]:
tpot_row = run_tpot(
    X_train_pre, y_train_np, X_test_pre, y_test_np,
    dataset_name="CoffeeShop",
    time_budget_seconds=3600,
    seed=42
)
tpot_row

Generation: : 74it [1:00:00, 48.65s/it]


{'dataset': 'CoffeeShop',
 'system': 'TPOT',
 'test_rmse': 206.40150515518218,
 'time_seconds': 3603.1635358333588,
 'details': 'search_space=linear-light'}

### Compile Framework Performance Metrics (Coffee Shop)

In [30]:
results_df = pd.DataFrame([my_row, flaml_row, tpot_row]).sort_values("test_rmse")
results_df

,dataset,system,test_rmse,time_seconds,details
1,CoffeeShop,FLAML,205.648558,3601.404152,best_estimator=catboost
2,CoffeeShop,TPOT,206.401505,3603.163536,search_space=linear-light
0,CoffeeShop,MyAutoML_LocalSearch,214.311383,3600.569636,"Config(algo='gbr', params={'n_estimators': 400..."


### Empirical Commentary: Coffee Shop Optimization Results

| AutoML Framework | Selected Best Pipeline Model Type | Wall-Clock Execution Time (s) | Out-of-Sample Test RMSE |
| :--- | :--- | :--- | :--- |
| **FLAML** | **CatBoost Regressor** | 3601.40s | **205.649** |
| **TPOT** | **Optimized Linear Pipeline Graph** | 3603.16s | **206.402** |
| **MyAutoML_LocalSearch** | **Gradient Boosting Regressor** (`max_depth=2`, `subsample=0.7`, `n_estimators=400`, `learning_rate=0.05`)| **3600.57s** | **214.311** |

#### Performance Analysis:
* On the Coffee Shop dataset, the performance order reversed. FLAML secured the optimal solution (**Test RMSE: 205.649**), closely shadowed by TPOT (**Test RMSE: 206.402**). Our custom search technique stabilized at a higher error floor (**Test RMSE: 214.311**).
* FLAML’s dominance here highlights the impact of expanding model choices. FLAML integrated a **CatBoost Regressor**, a specialized gradient boosting engine designed for categorical transformations that was not included in our custom optimizer's model space.
* While our local search engine correctly identified gradient boosting (`gbr`) as the best architecture for this data layout, its discrete search parameters limited its precision. It settled on a conservative tree structure (`max_depth=2`, `subsample=0.7`, `n_estimators=400`). This demonstrates the limitations of a standard discrete step grid when optimizing complex continuous spaces compared to FLAML's advanced cost-frugal Bayesian optimization strategies.